In [6]:
import random
from collections import Counter

# ----------------------------
# トランプ
# ----------------------------
class t_cards:
    def __init__(self):
        self.symbol = ['♠', '♥', '♦', '♣']
        self.number = ['A', '2', '3', '4', '5', '6', '7', '8',
                       '9', '10', 'J', 'Q', 'K']
        self.trump_card = [s + n for s in self.symbol for n in self.number]
        random.shuffle(self.trump_card)

    def draw(self):
        return self.trump_card.pop()


# ----------------------------
# プレイヤー（人間＆親(cpu)共通）
# ----------------------------
class player:
    def __init__(self, all_cards):
        self.all_cards = all_cards
        self.selected_cards = []

    # ★ 5枚引く
    def card_selection(self):
        for _ in range(5):
            self.selected_cards.append(self.all_cards.draw())

    # ★ プレイヤー用交換
    def change_card(self):
        print("現在の手札:", self.selected_cards)

        index = int(input(
            f"何番目のカードを捨てる？（1〜5）: "
            f"1={self.selected_cards[0]}, "
            f"2={self.selected_cards[1]}, "
            f"3={self.selected_cards[2]}, "
            f"4={self.selected_cards[3]}, "
            f"5={self.selected_cards[4]} "
        )) - 1

        if index < 0 or index > 4:
            print("無効な入力です。交換しません")
            return

        removed = self.selected_cards.pop(index)
        print("捨てたカード:", removed)

        new_card = self.all_cards.draw()
        self.selected_cards.append(new_card)

        print("引いたカード:", new_card)
        print("交換後の手札:", self.selected_cards)

    # ★ CPU(親もランダムでカード交換をして、交換したときには交換したことだけをPrintする)用
    def cpu_change_card(self, round_num):
      judge = card_judge(self.selected_cards)
      hand, score = judge.judge()

      # 強い役は交換しない
      if score >= 2:
        print(f"親は{round_num}回目で交換しませんでした")
        return

    # ワンペア
      if score == 1:
        counts = Counter(judge.values)
        pair_values = [v for v, c in counts.items() if c >= 2]

        for i, v in enumerate(judge.values):
            if v not in pair_values:
                self.selected_cards[i] = self.all_cards.draw()
                print(f"親は{round_num}回目で1枚交換しました")
                return

    # ハイカード
      if score == 0:
        min_value = min(judge.values)

        for i, v in enumerate(judge.values):
            if v == min_value:
                self.selected_cards[i] = self.all_cards.draw()
                print(f"親は{round_num}回目で1枚交換しました")
                return

# ----------------------------
# 役判定（そのまま）
# ----------------------------
class card_judge:
    def __init__(self, selected_cards):
        self.selected_cards = selected_cards

        self.suits = [c[0] for c in selected_cards]
        self.ranks = [c[1:] for c in selected_cards]

        rank_dict = {
            "2":2,"3":3,"4":4,"5":5,"6":6,"7":7,
            "8":8,"9":9,"10":10,"J":11,"Q":12,"K":13,"A":14
        }

        self.values = sorted([rank_dict[r] for r in self.ranks])

    def is_flush(self):
        return len(set(self.suits)) == 1

    def is_straight(self):
        v = self.values
        return v == list(range(v[0], v[0]+5))

    def judge(self):
        counts = sorted(Counter(self.values).values(), reverse=True)

        if self.is_flush() and self.is_straight():
            return "ストレートフラッシュ", 8
        elif counts == [4,1]:
            return "フォーカード", 7
        elif counts == [3,2]:
            return "フルハウス", 6
        elif self.is_flush():
            return "フラッシュ", 5
        elif self.is_straight():
            return "ストレート", 4
        elif counts == [3,1,1]:
            return "スリーカード", 3
        elif counts == [2,2,1]:
            return "ツーペア", 2
        elif counts == [2,1,1,1]:
            return "ワンペア", 1
        else:
            return "ハイカード", 0


# -----------------------
# ゲーム開始
# -----------------------

all_cards = t_cards()

player1 = player(all_cards)
cpu = player(all_cards)

# ★ ここでカード配る（重要！）
player1.card_selection()
cpu.card_selection()

print("あなたのカード:", player1.selected_cards)

# --- 交換処理（最大2回）---
for i in range(1, 3):
    decision = input(f"{i}回目：交換するなら1、しないなら2: ")

    if decision == "1":
        player1.change_card()
    else:
      print("入力が不正です。今回は交換なしで進みます。")

    cpu.cpu_change_card(i)

print("最終手札:", player1.selected_cards)
print("\n親のカード:", cpu.selected_cards)

# -----------------------
# 判定
# -----------------------
player_judge = card_judge(player1.selected_cards)
cpu_judge = card_judge(cpu.selected_cards)

player_hand, player_score = player_judge.judge()
cpu_hand, cpu_score = cpu_judge.judge()

print("\nあなたの役:", player_hand)
print("親の役:", cpu_hand)

# -----------------------
# 勝敗
# -----------------------
if player_score > cpu_score:
    print("あなたの勝ち！")
elif player_score < cpu_score:
    print("親の勝ち！")
else:
    print("引き分け！")

あなたのカード: ['♥5', '♥2', '♥3', '♥Q', '♣K']
1回目：交換するなら1、しないなら2: 1
現在の手札: ['♥5', '♥2', '♥3', '♥Q', '♣K']
何番目のカードを捨てる？（1〜5）: 1=♥5, 2=♥2, 3=♥3, 4=♥Q, 5=♣K 0
無効な入力です。交換しません
親は1回目で交換しませんでした
2回目：交換するなら1、しないなら2: 1
現在の手札: ['♥5', '♥2', '♥3', '♥Q', '♣K']
何番目のカードを捨てる？（1〜5）: 1=♥5, 2=♥2, 3=♥3, 4=♥Q, 5=♣K 5
捨てたカード: ♣K
引いたカード: ♥10
交換後の手札: ['♥5', '♥2', '♥3', '♥Q', '♥10']
親は2回目で交換しませんでした
最終手札: ['♥5', '♥2', '♥3', '♥Q', '♥10']

親のカード: ['♠4', '♥8', '♣4', '♣8', '♣A']

あなたの役: フラッシュ
親の役: ツーペア
あなたの勝ち！
